In [0]:
# Cell 1: Widget + create landing zones for the requested datasets only
dbutils.widgets.text("datasets", "ais_locations,port_calls,sea_state", "Datasets to fetch (comma-separated)")
selected_datasets = [d.strip() for d in dbutils.widgets.get("datasets").split(",")]

landing_base = "abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/landing"

for ds in selected_datasets:
    dbutils.fs.mkdirs(f"{landing_base}/{ds}")

print(f"Landing zone directories ready for: {selected_datasets}")

In [0]:
# Cell 2: Ingest the latest micro-batch — only for the selected datasets
import requests
import json
from datetime import datetime, timezone

landing_base = "abfss://maritime-lake@maritimepipeline.dfs.core.windows.net/landing"

all_endpoints = {
    "port_calls": "https://meri.digitraffic.fi/api/port-call/v1/port-calls",
    "sea_state": "https://meri.digitraffic.fi/api/sse/v1/measurements",
    "ais_locations": "https://meri.digitraffic.fi/api/ais/v1/locations"
}

endpoints = {k: v for k, v in all_endpoints.items() if k in selected_datasets}

headers = {
    "Accept-Encoding": "gzip",
    "Digitraffic-User": "Databricks-Portfolio-Project"
}

current_time = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

for data_type, url in endpoints.items():
    print(f"Fetching {data_type} from {url}...")

    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        json_string = json.dumps(data)
        file_path = f"{landing_base}/{data_type}/{current_time}.json"
        dbutils.fs.put(file_path, json_string, overwrite=True)
        print(f"Successfully saved {data_type} to {file_path}")
    else:
        print(f"Failed to fetch {data_type}. Status code: {response.status_code}")